# Block model QA/QC

This notebook runs the whole QA/QC workflow, one step at a time. Each step has a short note on what it does and what to look for.

**How to use it**
1. In **Step 0**, choose demo mode (a made-up model with planted problems) or set the path to your project YAML file.
2. Run the cells from top to bottom: **Run → Run All Cells** in JupyterLab, or **Run All** in VS Code.
3. If a step stops with a message, fix the problem (usually in the project YAML), then run everything again from the top.

All the calculations live in the `qaqc` package. This notebook only runs the steps and shows the results, so you don't need to edit anything below Step 0.

## Step 0: Choose what to run

In [ ]:
# True: try the notebook on a small made-up model (no real data needed).
# False: use your own project YAML file below.
DEMO = True

# Your project settings. Copy config/project_example.yaml, edit it, and put its path
# here. Forward slashes work on Windows too, e.g. "C:/Projects/MRE/qaqc.yaml".
# A relative path is read from the notebooks/ folder.
CONFIG_PATH = "../config/my_project.yaml"

# Reuse the fast copy of the block model saved by an earlier run (re-read if the CSV
# has changed). Set to False to always read the CSV again.
USE_CACHE = True

In [ ]:
from IPython.display import display

from qaqc import display as show
from qaqc.demo import REPO_ROOT, make_demo_project
from qaqc.modules import classification, composites, density, geometry, grades, support

show.setup_notebook()
if DEMO:
    CONFIG_PATH = make_demo_project(REPO_ROOT / "demo_project")
    print(f"Demo mode: made-up project written to {CONFIG_PATH.parent}")
else:
    print(f"Using {CONFIG_PATH}")

## Step 1: Project settings

Loads the project YAML and checks every setting. A misspelled or invalid setting stops here with a message saying which one. Check that the input files say **found**.

In [ ]:
cfg = show.load_config(CONFIG_PATH)
display(show.config_summary(cfg))

## Step 2: Column matching

Works out which column in the CSV is which (coordinates, block sizes, domain, density, grades and so on). Only the top of the file is read here.

**Check:** every row in the table points at the right column. If one is wrong or missing, set it under `column_overrides` in the project YAML.

In [ ]:
s = show.match_columns(cfg)
display(show.file_summary(s))
display(s.describe())

Columns in the file that no check uses, and notes from matching:

In [ ]:
display(show.unused_columns(s))
display(show.schema_notes(s))

## Step 3: Read and clean the block model

Reads the whole file, treats placeholder values (such as -99 or `Outside`) as missing, leaves out deselected rows, and sets up the model grid.

**Check:** the row counts look right, and the grid values (cell size, corner) match the model. *Where it came from* says whether each value was read from the file, the settings, or worked out from the blocks.

In [ ]:
ctx = show.read_model(cfg, use_cache=USE_CACHE)
display(show.reading_summary(ctx))
display(show.grid_summary(ctx))

## Step 4: Geometry

Checks the blocks against the grid: duplicate block centres, blocks off the grid, sub-blocks that overfill their parent cell, and percentages outside 0-100%. Works out block volumes and tonnes.

In [ ]:
geometry.run(ctx)
display(show.geometry_summary(ctx))
display(show.flag_table(ctx, "geometry"))

Sub-blocked models only: do sub-blocks in the same parent cell carry different grades? Wireframe volumes are compared only if `target_wireframe_volumes` is set. Empty tables mean the check does not apply.

In [ ]:
display(show.subblock_support_table(ctx))
display(show.wireframe_volume_table(ctx))
display(show.warnings_table(ctx, "geometry"))

## Step 5: Density

Flags blocks with missing density or density outside the allowed range (`density` in the project YAML). Shows density by domain and how density relates to grade.

In [ ]:
density.run(ctx)
display(show.density_summary(ctx))
display(show.density_by_domain(ctx))

In [ ]:
display(show.density_vs_grade(ctx))
display(show.warnings_table(ctx, "density"))

## Step 6: Grades

For each element: the unit, which domains it is estimated in, and blocks with a negative grade, a grade above the ceiling, a missing grade where one is expected, or a grade above the diagnostic cutoff in waste (`elements` in the project YAML).

**Check:** the unit is right (a *LIKELY WRONG* note means the values don't fit the unit), and the domains match where you estimated each element.

In [ ]:
grades.run(ctx)
display(show.grades_summary(ctx))
display(show.warnings_table(ctx, "grades"))

Grade statistics by domain, weighted by tonnes. Change the element to see another one.

In [ ]:
display(show.grades_by_domain(ctx, "cu"))

## Step 7: Classification and resource statement

Maps class codes to Measured / Indicated / Inferred / Unclassified, totals tonnes, grade and metal by class and domain, and flags unclassified blocks above the diagnostic cutoff. If `reporting: resource_cutoff` is set, the resource statement is reproduced so you can check it against the published one.

In [ ]:
classification.run(ctx)
display(show.skipped_note(ctx, "classification"))
display(show.classification_summary(ctx))
display(show.unmapped_codes(ctx))
display(show.resource_statement(ctx))

Inventory by domain and class:

In [ ]:
display(show.classification_inventory(ctx))

## Step 8: Estimation support

Uses the estimation details in the model (search pass, samples, holes, slope of regression, kriging efficiency, distances) to find Measured and Indicated blocks that fail the project's classification criteria (`support: thresholds_by_rank`).

In [ ]:
support.run(ctx)
display(show.support_low(ctx))
display(show.support_passes(ctx))

In [ ]:
display(show.support_stats(ctx))

## Step 9: Composites, bias, grade-tonnage curves and swaths

Reads the composites (skipped if `paths: composites_csv` is null), checks each composite sits in a block of the same domain, and compares the block model with the composites and a nearest-neighbour model.

**Check:** the bias table. Amber and red rows differ from the nearest-neighbour model by more than `composites: bias_warn_pct` / `bias_fail_pct`. Grade-tonnage curves and swath plots are in the HTML report.

In [ ]:
composites.run(ctx)
display(show.composites_summary(ctx))
display(show.bias_table(ctx))

Composite domain (rows) against the domain of the block it sits in (columns):

In [ ]:
display(show.domain_matrix(ctx))
display(show.warnings_table(ctx, "composites"))

## Step 10: Write the report and output files

Writes into the output folder (`paths: output_dir`):
- **BM_QAQC_Report.html**: the full report with all tables and charts (grade-tonnage curves, swaths and more). Opens in any browser, works offline.
- **flagged_blocks_leapfrog.csv**: every flagged block, to import into Leapfrog as points. The report's *Flag register* tab gives a filter for each flag.
- **composite_domain_mismatch.csv**, **flagged_composites.csv**: composites outside the model or in the wrong domain.
- **audit_bundle.json**: every result in one file, for records and comparisons.

Close these files in Excel first: Windows will not let them be replaced while open.

In [ ]:
display(show.write_outputs(ctx))

## Summary

Every message from the run, worst first: **red** = a result is missing or not trustworthy, **amber** = worth checking, white = for information.

In [ ]:
display(show.warnings_table(ctx))

Flagged blocks for every check, and how long each step took:

In [ ]:
display(show.flag_table(ctx))
display(show.timings(ctx))